In [7]:
from pyspark.sql import SparkSession
from pyspark.sql.functions import col, trim, lower, lit, when, coalesce, concat, split, regexp_replace, upper
from delta import configure_spark_with_delta_pip

# 1. SETUP
warehouse_location = "hdfs://hdfs-nn:9000/warehouse"
builder = SparkSession.builder.appName("Gold_Fact_Movie_Ratings_Final_Fixed") \
    .config("spark.sql.warehouse.dir", warehouse_location) \
    .config("spark.sql.extensions", "io.delta.sql.DeltaSparkSessionExtension") \
    .config("spark.sql.catalog.spark_catalog", "org.apache.spark.sql.delta.catalog.DeltaCatalog") \
    .config("spark.jars.packages", "io.delta:delta-core_2.12:2.4.0") \
    .enableHiveSupport()
spark = configure_spark_with_delta_pip(builder).getOrCreate()

# 2. CARREGAR DADOS DE SUPORTE (Dimensões e País)
print("🌍 A preparar dados...")

# Dimensão Filme Base
df_dim_base = spark.read.format("delta").load(f"{warehouse_location}/gold.db/dim_filme")
df_dim_filme = df_dim_base.select("id_filme", trim(lower(col("title"))).alias("match_title"))

# Dimensão País
df_dim_paises = spark.read.format("delta").load(f"{warehouse_location}/gold.db/dim_paises") \
    .select(col("id_pais"), col("sigla"), col("nome_completo"))

# --- NOVO: Carregar Dimensão Calendário ---
df_dim_calendario = spark.read.format("delta").load(f"{warehouse_location}/gold.db/dim_calendario") \
    .select(col("id_calendario"), col("ano"))

# 3. DETERMINAR O PAÍS PRINCIPAL DE CADA FILME
# Vamos usar a Amazon como fonte principal. Se for lista ['US', 'UK'], ficamos só com 'US'.
df_pais_source = spark.read.format("delta").load(f"{warehouse_location}/silver.db/titles") \
    .select(trim(lower(col("title"))).alias("match_title"), col("production_countries"))

df_pais_clean = df_pais_source \
    .withColumn("pais_raw", regexp_replace(col("production_countries"), r"[\[\]']", "")) \
    .withColumn("primeiro_pais", split(col("pais_raw"), ",").getItem(0)) \
    .withColumn("primeiro_pais", upper(trim(col("primeiro_pais")))) \
    .filter(col("primeiro_pais") != "")

# Join para obter o ID do País (P1...)
df_mapa_pais = df_pais_clean.join(df_dim_filme, "match_title") \
    .join(df_dim_paises, (df_pais_clean.primeiro_pais == df_dim_paises.sigla) | (df_pais_clean.primeiro_pais == df_dim_paises.nome_completo)) \
    .select("id_filme", "id_pais") \
    .dropDuplicates(["id_filme"]) # Garante 1:1

# 4. PREPARAR RATINGS
df_amazon = spark.read.format("delta").load(f"{warehouse_location}/silver.db/titles") \
    .select(trim(lower(col("title"))).alias("t_ama"), col("release_year").alias("y_ama"), (col("imdb_score") * 10).alias("score_amazon"))
df_netflix = spark.read.format("delta").load(f"{warehouse_location}/silver.db/netflix_titles") \
    .select(trim(lower(col("title"))).alias("t_net"), col("release_year").alias("y_net"), (col("imdb_score") * 10).alias("score_netflix"))
df_rotten = spark.read.format("delta").load(f"{warehouse_location}/silver.db/rotten_tomatoes") \
    .select(trim(lower(col("movie_title"))).alias("t_rot"), col("release_year").alias("y_rot"), col("tomatometer_rating"), col("audience_rating"))

# Joins com Dim Filme para ter IDs
fact_ama = df_amazon.join(df_dim_filme, df_amazon.t_ama == df_dim_filme.match_title).select("id_filme", "y_ama", "score_amazon")
fact_net = df_netflix.join(df_dim_filme, df_netflix.t_net == df_dim_filme.match_title).select("id_filme", "y_net", "score_netflix")
fact_rot = df_rotten.join(df_dim_filme, df_rotten.t_rot == df_dim_filme.match_title).select("id_filme", "y_rot", "tomatometer_rating", "audience_rating")

# Consolidar (Full Join)
df_join = fact_ama.join(fact_net, ["id_filme"], "full_outer") \
    .join(fact_rot, ["id_filme"], "full_outer")

# Gerar Colunas Finais
df_final = df_join.select(
    col("id_filme"),
    coalesce(fact_ama.y_ama, fact_net.y_net, fact_rot.y_rot).alias("ano_final"),
    col("score_amazon"),
    col("score_netflix"),
    col("tomatometer_rating").alias("score_rotten_critics"),
    col("audience_rating").alias("score_rotten_audience")
)

# --- AQUI ESTAVA O ERRO ---
# Removemos o concat("C"). Agora fazemos JOIN com a dimensão para buscar o ID numérico.
print("🔗 A ligar ao Calendário...")
df_final_com_cal = df_final.join(df_dim_calendario, df_final.ano_final == df_dim_calendario.ano, "left")

# 5. JUNTAR TUDO: NOME DO FILME + ID PAÍS
print("🔗 A montar a tabela final...")

df_gold = df_final_com_cal \
    .join(df_dim_base.select("id_filme", "title"), "id_filme", "left") \
    .join(df_mapa_pais, "id_filme", "left") \
    .select(
        col("id_filme"), 
        col("title"), 
        col("id_pais"),          
        col("id_calendario"), # Agora vem da dimensão (número inteiro)
        col("score_amazon"), 
        col("score_netflix"), 
        col("score_rotten_critics"), 
        col("score_rotten_audience")
    )

# 6. GRAVAR
fact_path = f"{warehouse_location}/gold.db/fact_movie_ratings"
print(f"💾 A gravar em: {fact_path}")

df_gold.write.format("delta").mode("overwrite").option("overwriteSchema", "true").save(fact_path)
spark.sql(f"CREATE TABLE IF NOT EXISTS gold.fact_movie_ratings USING DELTA LOCATION '{fact_path}'")

print("✅ Tabela criada! IDs corrigidos (Inteiros em Filme, País e Calendário).")
df_gold.orderBy("score_amazon", ascending=False).show(5)

🌍 A preparar dados...
🔗 A ligar ao Calendário...
🔗 A montar a tabela final...
💾 A gravar em: hdfs://hdfs-nn:9000/warehouse/gold.db/fact_movie_ratings
✅ Tabela criada! IDs corrigidos (Inteiros em Filme, País e Calendário).
+--------+--------------------+-------+-------------+------------+-------------+--------------------+---------------------+
|id_filme|               title|id_pais|id_calendario|score_amazon|score_netflix|score_rotten_critics|score_rotten_audience|
+--------+--------------------+-------+-------------+------------+-------------+--------------------+---------------------+
|   20383|          Pawankhind|     67|          123|        99.0|         null|                null|                 null|
|   33429|Water Helps the B...|   null|          120|        97.0|         null|                null|                 null|
|    6551|   Couple of Mirrors|     29|          122|        95.0|         null|                null|                 null|
|   25558|          Tari Sathe|   

In [8]:
from pyspark.sql import SparkSession
from pyspark.sql.functions import col
from delta import configure_spark_with_delta_pip

# ===================================================
# 1. SETUP
# ===================================================
# Iniciar Spark (caso não esteja iniciado)
builder = SparkSession.builder \
    .appName("Analise_Qualidade_Filmes") \
    .config("spark.sql.warehouse.dir", "hdfs://hdfs-nn:9000/warehouse") \
    .config("spark.sql.extensions", "io.delta.sql.DeltaSparkSessionExtension") \
    .config("spark.sql.catalog.spark_catalog", "org.apache.spark.sql.delta.catalog.DeltaCatalog") \
    .config("spark.jars.packages", "io.delta:delta-core_2.12:2.4.0") \
    .enableHiveSupport()

spark = configure_spark_with_delta_pip(builder).getOrCreate()

warehouse_location = "hdfs://hdfs-nn:9000/warehouse"

# CORREÇÃO 1: O caminho correto da tua tabela final
path_gold_fact = f"{warehouse_location}/gold.db/fact_movie_ratings"

print(f"📖 A ler: {path_gold_fact}")
df_completo = spark.read.format("delta").load(path_gold_fact)

# CORREÇÃO 2: Removi o JOIN com a dim_filme.
# Porquê? A fact_movie_ratings JÁ TEM o título! Fazer join duplicaria a coluna e daria erro.

# ===================================================
# 2. PROCURAR INTERSEÇÕES (As "Agulhas no Palheiro")
# ===================================================

print("\n=== 🔎 PROVA 1: Filmes na AMAZON que também têm ROTTEN TOMATOES ===")
df_completo.filter(
    col("score_amazon").isNotNull() & 
    col("score_rotten_critics").isNotNull()
).select("title", "score_amazon", "score_rotten_critics").show(10, truncate=False)

print("\n=== 🔎 PROVA 2: Filmes na NETFLIX que também têm ROTTEN TOMATOES ===")
df_completo.filter(
    col("score_netflix").isNotNull() & 
    col("score_rotten_critics").isNotNull()
).select("title", "score_netflix", "score_rotten_critics").show(10, truncate=False)

📖 A ler: hdfs://hdfs-nn:9000/warehouse/gold.db/fact_movie_ratings

=== 🔎 PROVA 1: Filmes na AMAZON que também têm ROTTEN TOMATOES ===
+------------------------+------------+--------------------+
|title                   |score_amazon|score_rotten_critics|
+------------------------+------------+--------------------+
|An American Rhapsody    |68.0        |55.0                |
|Dead Ant                |48.0        |64.0                |
|I Hate Kids             |51.0        |22.0                |
|No Tears For The Dead   |67.0        |33.0                |
|Space Truckers          |53.0        |8.0                 |
|The Cheap Detective     |64.0        |47.0                |
|The Phantom of the Opera|75.0        |33.0                |
|The Phantom of the Opera|75.0        |38.0                |
|The Ugly Truth          |64.0        |14.0                |
|Afternoon Delight       |57.0        |67.0                |
+------------------------+------------+--------------------+
only showing

In [9]:
from pyspark.sql import SparkSession
from pyspark.sql.functions import col, row_number, count
from pyspark.sql.window import Window

# ===================================================
# 1. SETUP
# ===================================================
warehouse_location = "hdfs://hdfs-nn:9000/warehouse"
path_gold_fact = f"{warehouse_location}/gold.db/fact_classificacao_filmes"
path_gold_dim = f"{warehouse_location}/gold.db/dim_filme"

df_fact = spark.read.format("delta").load(path_gold_fact)
df_dim = spark.read.format("delta").load(path_gold_dim)

# Juntar Nomes
df_completo = df_fact.join(df_dim, "id_filme", "left") \
    .select(
        "title", 
        "score_amazon", 
        "score_netflix", 
        "score_rotten_critics", 
        "score_rotten_audience"
    )

# ===================================================
# 2. LIMPEZA DE DUPLICADOS (O "Home" Fix)
# ===================================================
# Definimos uma regra: Se houver nomes repetidos, ficamos com aquele que tem
# a maior nota no Rotten Tomatoes (assumindo ser o filme "famoso")
window_spec = Window.partitionBy("title").orderBy(col("score_rotten_critics").desc())

df_limpo = df_completo.withColumn("rank", row_number().over(window_spec)) \
    .filter(col("rank") == 1) \
    .drop("rank")

# ===================================================
# 3. MOSTRAR O "SANTO GRAAL" LIMPO
# ===================================================
print("\n=== 🏆 Filmes confirmados nas 3 plataformas ===")
# Filtramos apenas os que têm notas nas 3 colunas
df_santo_graal = df_limpo.filter(
    col("score_amazon").isNotNull() & (col("score_amazon") > 0) &
    col("score_netflix").isNotNull() & (col("score_netflix") > 0) &
    col("score_rotten_critics").isNotNull()
)

df_santo_graal.select("title", "score_amazon", "score_netflix", "score_rotten_critics") \
    .orderBy(col("score_rotten_critics").desc()) \
    .show(20, truncate=False)


=== 🏆 Filmes confirmados nas 3 plataformas ===
+-----------------------------------------+------------+-------------+--------------------+
|title                                    |score_amazon|score_netflix|score_rotten_critics|
+-----------------------------------------+------------+-------------+--------------------+
|City of Joy                              |65.0        |75.0         |100.0               |
|Conspiracy                               |46.0        |62.0         |100.0               |
|Heidi                                    |74.0        |71.0         |100.0               |
|Kabhi Khushi Kabhie Gham                 |74.0        |74.0         |100.0               |
|Mike Birbiglia: My Girlfriend's Boyfriend|80.0        |80.0         |100.0               |
|Sahara                                   |67.0        |55.0         |100.0               |
|The Black Godfather                      |44.0        |74.0         |100.0               |
|The Hospital                   